# 9.1 模糊逻辑分类器

**参考文献**：Ryzhkov & Zrnic, Chapter 9, pp. 861-900

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def membership_function(x, center, width, type='gaussian'):
    """
    模糊隶属度函数
    """
    if type == 'gaussian':
        return np.exp(-((x - center) / width)**2)
    elif type == 'triangle':
        return np.maximum(0, 1 - np.abs(x - center) / width)
    return np.zeros_like(x)

def fuzzy_classify(Z, ZDR, rho_hv):
    """
    模糊逻辑水凝物分类
    """
    classes = ['雨', '雪', '冰雹', '地物']
    scores = {c: 0 for c in classes}
    
    # 雨：中等ZDR，高ρhv
    scores['雨'] = (membership_function(ZDR, 2, 1.5) + 
                   membership_function(rho_hv, 0.97, 0.05)) / 2
    
    # 雪：低ZDR，中等ρhv
    scores['雪'] = (membership_function(ZDR, 0.5, 2) + 
                   membership_function(rho_hv, 0.93, 0.1)) / 2
    
    # 冰雹：高ZDR，低ρhv
    scores['冰雹'] = (membership_function(ZDR, 3.5, 1.5) + 
                      (1 - membership_function(rho_hv, 0.85, 0.1))) / 2
    
    # 地物：低ρhv
    scores['地物'] = (1 - membership_function(rho_hv, 0.75, 0.15))
    
    return scores

def plot_fuzzy_classifier():
    """模糊分类器可视化"""
    ZDR_range = np.linspace(-2, 6, 100)
    rho_range = np.linspace(0.5, 1.0, 100)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 雨隶属度
    ax1 = axes[0, 0]
    ax1.plot(ZDR_range, membership_function(ZDR_range, 2, 1.5), 'b-', linewidth=2, label='ZDR')
    ax1.plot(rho_range, membership_function(rho_range, 0.97, 0.05), 'g-', linewidth=2, label='ρhv')
    ax1.set_xlabel('变量值', fontsize=11)
    ax1.set_ylabel('隶属度', fontsize=11)
    ax1.set_title('雨的隶属度函数', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 分类决策区域
    ax2 = axes[0, 1]
    ZDR_mesh, rho_mesh = np.meshgrid(np.linspace(-2, 6, 50), np.linspace(0.5, 1.0, 50))
    ZDR_flat = ZDR_mesh.flatten()
    rho_flat = rho_mesh.flatten()
    
    # 为每个点分类
    colors = []
    for zdr, rho in zip(ZDR_flat, rho_flat):
        scores = fuzzy_classify(40, zdr, rho)  # 固定Z=40
        best_class = max(scores, key=scores.get)
        colors.append(best_class)
    
    color_map = {'雨': 'blue', '雪': 'cyan', '冰雹': 'red', '地物': 'brown'}
    ax2.scatter(ZDR_flat, rho_flat, c=[color_map[c] for c in colors], s=20, alpha=0.5)
    ax2.set_xlabel('ZDR (dB)', fontsize=11)
    ax2.set_ylabel('|ρhv|', fontsize=11)
    ax2.set_title('水凝物分类决策区域 (Z=40dBZ)', fontsize=12)
    ax2.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=v, label=k, markersize=10) for k, v in color_map.items()])
    ax2.grid(True, alpha=0.3)
    
    # 各类型阈值
    ax3 = axes[1, 0]
    ax3.axis('off')
    table_data = [
        ['类型', 'ZDR范围', 'ρhv范围', '典型特征'],
        ['雨', '1-4 dB', '>0.95', '下落雨滴'],
        ['雪', '-1-2 dB', '0.85-0.95', '冰晶聚合体'],
        ['冰雹', '>3 dB', '<0.85', '大冰粒/融化'],
        ['地物', '不定', '<0.75', '固定目标']
    ]
    table = ax3.table(cellText=table_data, loc='center', cellLoc='left', colWidths=[0.15, 0.2, 0.2, 0.35])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 2)
    for i in range(4):
        table[(0, i)].set_facecolor('lightgray')
    ax3.set_title('水凝物分类阈值', fontsize=12, pad=20)
    
    # 分类示例
    ax4 = axes[1, 1]
    samples = [(35, 2.5, 0.97, '雨'), (25, 0.5, 0.90, '雪'), 
               (50, 4.0, 0.80, '冰雹'), (45, 1.0, 0.60, '地物')]
    for Z, ZDR, rho, label in samples:
        scores = fuzzy_classify(Z, ZDR, rho)
        ax4.bar(label, scores[label], color=color_map[label], alpha=0.7)
        ax4.text(samples.index((Z, ZDR, rho, label)), scores[label] + 0.05, 
                f'{scores[label]:.2f}', ha='center')
    ax4.set_ylabel('分类置信度', fontsize=11)
    ax4.set_title('样本分类结果', fontsize=12)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 模糊逻辑分类器 ===")
    print("隶属度函数定义了每个变量属于某类型的程度")
    print("综合得分最高者获胜")
    print("可调整阈值以优化分类效果")

In [ ]:
plot_fuzzy_classifier()

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 9, Springer.